# 🤖 Generative AI Modules
Three tools built on **Gemini Pro**, all traced with Langfuse:

1. **Chatbot** — Multi-turn conversation with memory and configurable system prompt
2. **PDF Assistant** — RAG pipeline: chunk → embed → FAISS → retrieve → generate
3. **Medical Tool** — Symptom checker with structured output, guardrails, and emergency detection

**Prerequisites:** Set your `GEMINI_API_KEY` below.

In [ ]:
!git clone https://github.com/Aurovindhya/ml-portfolio-suite.git
%cd ml-portfolio-suite
!pip install -q google-generativeai faiss-cpu pypdf langfuse python-dotenv

In [ ]:
import os
# Set your API key here or mount Google Drive and load from .env
os.environ['GEMINI_API_KEY'] = 'your_gemini_api_key_here'

# Optional Langfuse tracing
# os.environ['LANGFUSE_PUBLIC_KEY'] = 'pk-lf-...'
# os.environ['LANGFUSE_SECRET_KEY'] = 'sk-lf-...'

## 1. Chatbot — Multi-turn conversation

In [ ]:
import sys; sys.path.insert(0, '.')
from modules.generative.chatbot.model import ChatSession

session = ChatSession(system_prompt="You are a concise ML tutor. Answer in 2-3 sentences max.")

questions = [
    'What is transfer learning?',
    'How does it differ from fine-tuning?',
    'Give me a one-line example using EfficientNet.',
]

for q in questions:
    result = session.chat(q)
    print(f'Q: {q}')
    print(f'A: {result["reply"]}')
    print(f'   [{result["input_tokens"]} in / {result["output_tokens"]} out | {result["inference_ms"]:.0f}ms | trace: {result["trace_id"]}]')
    print()

## 2. PDF Assistant — RAG pipeline

In [ ]:
# Create a sample PDF to test with
try:
    from fpdf import FPDF
except ImportError:
    import subprocess; subprocess.run(['pip', 'install', '-q', 'fpdf2'])
    from fpdf import FPDF

pdf = FPDF()
pdf.add_page()
pdf.set_font('Helvetica', size=12)
content = [
    'Machine Learning Portfolio — Technical Summary',
    '',
    'This document describes a full-stack ML platform built with PyTorch, FastAPI, and Gemini Pro.',
    'The platform covers regression, classification, generative AI, computer vision, and agentic workflows.',
    '',
    'Regression modules use XGBoost for flight fare prediction, achieving MAE of approximately 1200 INR.',
    'House price prediction uses a Ridge, Lasso, and ElasticNet stacked ensemble trained on the Ames Housing dataset.',
    'Gold price prediction uses Random Forest with RSI and MACD technical indicators as features.',
    '',
    'All generative AI modules are traced with Langfuse for token counts, latency, and evaluation scores.',
]
for line in content:
    pdf.cell(0, 10, line, ln=True)

pdf_path = '/tmp/sample_portfolio.pdf'
pdf.output(pdf_path)
print(f'Sample PDF created: {pdf_path}')

In [ ]:
from modules.generative.pdf_assistant.model import PDFAssistant

assistant = PDFAssistant()
with open(pdf_path, 'rb') as f:
    assistant.ingest(f.read(), doc_name='portfolio_summary.pdf')

print(f'Indexed {len(assistant._chunks)} chunks from PDF')

questions = [
    'What regression models are used in this platform?',
    'How is observability handled for the generative AI modules?',
    'What is the MAE for flight fare prediction?',
]

for q in questions:
    result = assistant.ask(q)
    print(f'Q: {q}')
    print(f'A: {result["answer"]}')
    print(f'   [chunks used: {result["chunks_used"]} | {result["inference_ms"]:.0f}ms | trace: {result["trace_id"]}]')
    print()

## 3. Medical Tool — Structured symptom analysis

In [ ]:
from modules.generative.medical_tool.model import analyze_symptoms

symptoms = ['persistent cough', 'fever', 'fatigue', 'shortness of breath']
result   = analyze_symptoms(symptoms, age=45, gender='male')

print('=== MEDICAL BRIEF ===')
print(f'Emergency: {result.is_emergency}')
print(f'Recommended action: {result.recommended_action}')
print()
print('Possible conditions:')
for cond in result.possible_conditions:
    print(f'  [{cond.urgency.upper()}] {cond.name}: {cond.match_reason}')
print()
print('Red flags:', result.red_flags)
print()
print(result.disclaimer)
print(f'\n[trace: {result.trace_id} | {result.inference_ms:.0f}ms]')

In [ ]:
# Emergency detection test
emergency_symptoms = ['chest pain', 'left arm pain', 'difficulty breathing']
result = analyze_symptoms(emergency_symptoms)

print(f'Emergency detected: {result.is_emergency}')
print(f'Action: {result.recommended_action}')